# Notebook 2: Kernel Patterns

Run three kernels and connect each one to a different idea: a grid-stride loop, shared-memory reduction, and two-dimensional tiling.


In [ ]:
from pathlib import Path
import subprocess

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "CMakeLists.txt").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the cuda-kernels-a100-beginners repository")

ROOT = find_repo_root(Path.cwd())
subprocess.run(["cmake", "-S", str(ROOT), "-B", str(ROOT / "build"), "-DCMAKE_CUDA_ARCHITECTURES=80"], check=True)
subprocess.run(["cmake", "--build", str(ROOT / "build"), "-j"], check=True)


In [ ]:
targets = ["07_grid_stride", "08_reduction", "09_tiled_matmul"]
outputs = {}
for target in targets:
    result = subprocess.run([str(ROOT / "build" / target)], text=True, capture_output=True)
    outputs[target] = result.stdout.strip()
    print(target, "->", result.stdout.strip())
    if result.stderr:
        print(result.stderr)
    assert result.returncode == 0
    assert "PASS" in result.stdout


## Focused Code Reading

The next cell prints only lines containing important primitives. Then open the complete files in an editor.


In [ ]:
needles = ("gridDim", "__shared__", "__syncthreads", "dim3", "blockIdx")
for target in targets:
    path = ROOT / "lessons" / f"{target}.cu"
    print(f"\n== {path.name} ==")
    for number, line in enumerate(path.read_text().splitlines(), 1):
        if any(needle in line for needle in needles):
            print(f"{number:3}: {line}")


## Experiment

Change only one variable at a time:

- Number of blocks in the grid-stride loop
- Number of threads in the reduction
- `TILE` size in matrix multiplication

After every change: build, verify `PASS`, and only then measure.
